# Experimento B: Lectura monolítica vs. Lectura incremental

**Objetivo:** Comparar el rendimiento, consumo de memoria y estabilidad entre:
1. Lectura tradicional: `pd.read_csv()` (completo en memoria)
2. Lectura incremental: `pd.read_csv(..., chunksize=n)` (por lotes)

**Métricas evaluadas:**
- Tiempo total de lectura
- Pico de uso de memoria
- Número de registros procesados
- Estabilidad (éxito/fallo)
- Límites prácticos

In [7]:
import pandas as pd
import numpy as np
import time
import tracemalloc
import os

# Ruta del archivo
ruta_csv = "../dataset/dataset_tienda.csv"

# Verificar tamaño del archivo
tamaño_mb = os.path.getsize(ruta_csv) / (1024 * 1024)
print(f"Tamaño del archivo: {tamaño_mb:.2f} MB")
print("-" * 60)

# ============================================================
# 1. LECTURA MONOLÍTICA (Completo en memoria)
# ============================================================
print("LECTURA 1: Monolítica (pd.read_csv completo)")
print("-" * 60)

tracemalloc.start()
tiempo_inicio_mono = time.time()

try:
    dataset_mono = pd.read_csv(ruta_csv)
    tiempo_fin_mono = time.time()
    
    # Capturar uso de memoria
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    tiempo_lectura_mono = tiempo_fin_mono - tiempo_inicio_mono
    print(f"✓ Lectura exitosa")
    print(f"  Filas: {len(dataset_mono):,}")
    print(f"  Columnas: {dataset_mono.shape[1]}")
    print(f"  Tiempo: {tiempo_lectura_mono:.2f} segundos")
    print(f"  Memoria pico: {peak / (1024 * 1024):.2f} MB")
    print(f"  Primeras filas:")
    display(dataset_mono.head(3))
    
    lectura_mono_exitosa = True
    
except MemoryError:
    lectura_mono_exitosa = False
    tiempo_lectura_mono = None
    peak = None
    print("✗ Error: Memoria insuficiente para cargar el archivo completo")
    print("  El sistema no puede asignar memoria suficiente")
    tracemalloc.stop()
    
except Exception as e:
    lectura_mono_exitosa = False
    tiempo_lectura_mono = None
    peak = None
    print(f"✗ Error durante la lectura: {type(e).__name__}: {e}")
    tracemalloc.stop()


Tamaño del archivo: 464.50 MB
------------------------------------------------------------
LECTURA 1: Monolítica (pd.read_csv completo)
------------------------------------------------------------
✓ Lectura exitosa
  Filas: 10,000,000
  Columnas: 6
  Tiempo: 5.44 segundos
  Memoria pico: 863.94 MB
  Primeras filas:


,id_usuario,objeto,precio_venta,vendedor,fecha_venta,canal_venta
0,180325,Consola,2580484,Luis,2023-03-01,online
1,796559,Libro,48995,Pedro,2025-03-03,online
2,689113,Tablet,557920,Miguel,2025-05-11,online


In [8]:
print("\n" + "="*60)
print("LECTURA 2: Incremental (pd.read_csv con chunksize)")
print("-" * 60)

# Definir tamaño de chunks
tamaño_chunk = 50000  # 50k filas por chunk

tracemalloc.start()
tiempo_inicio_inc = time.time()

chunks_procesados = 0
filas_totales_inc = 0
datos_chunks = []

try:
    # Usar reader con chunksize
    lector = pd.read_csv(ruta_csv, chunksize=tamaño_chunk)
    
    for chunk in lector:
        chunks_procesados += 1
        filas_totales_inc += len(chunk)
        datos_chunks.append(chunk)
        
        # Mostrar progreso cada 10 chunks
        if chunks_procesados % 10 == 0:
            current, _ = tracemalloc.get_traced_memory()
            print(f"  Procesados: {chunks_procesados} chunks, {filas_totales_inc:,} filas "
                  f"(Memoria actual: {current / (1024 * 1024):.2f} MB)")
    
    tiempo_fin_inc = time.time()
    current, peak_inc = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    tiempo_lectura_inc = tiempo_fin_inc - tiempo_inicio_inc
    
    print(f"\n✓ Lectura incremental completada")
    print(f"  Chunks totales: {chunks_procesados}")
    print(f"  Tamaño por chunk: {tamaño_chunk:,} filas")
    print(f"  Filas procesadas: {filas_totales_inc:,}")
    print(f"  Tiempo total: {tiempo_lectura_inc:.2f} segundos")
    print(f"  Memoria pico: {peak_inc / (1024 * 1024):.2f} MB")
    
    # Recombinar chunks para verificación
    dataset_incremental = pd.concat(datos_chunks, ignore_index=True)
    print(f"  Primeras filas (concatenadas):")
    display(dataset_incremental.head(3))
    
    lectura_inc_exitosa = True
    
except Exception as e:
    lectura_inc_exitosa = False
    tiempo_lectura_inc = None
    peak_inc = None
    print(f"✗ Error durante la lectura incremental: {type(e).__name__}: {e}")
    tracemalloc.stop()



LECTURA 2: Incremental (pd.read_csv con chunksize)
------------------------------------------------------------
  Procesados: 10 chunks, 500,000 filas (Memoria actual: 24.43 MB)
  Procesados: 20 chunks, 1,000,000 filas (Memoria actual: 47.63 MB)
  Procesados: 30 chunks, 1,500,000 filas (Memoria actual: 71.77 MB)
  Procesados: 40 chunks, 2,000,000 filas (Memoria actual: 94.97 MB)
  Procesados: 50 chunks, 2,500,000 filas (Memoria actual: 119.13 MB)
  Procesados: 60 chunks, 3,000,000 filas (Memoria actual: 142.32 MB)
  Procesados: 70 chunks, 3,500,000 filas (Memoria actual: 166.48 MB)
  Procesados: 80 chunks, 4,000,000 filas (Memoria actual: 189.66 MB)
  Procesados: 90 chunks, 4,500,000 filas (Memoria actual: 213.34 MB)
  Procesados: 100 chunks, 5,000,000 filas (Memoria actual: 237.01 MB)
  Procesados: 110 chunks, 5,500,000 filas (Memoria actual: 260.69 MB)
  Procesados: 120 chunks, 6,000,000 filas (Memoria actual: 284.85 MB)
  Procesados: 130 chunks, 6,500,000 filas (Memoria actual: 308

,id_usuario,objeto,precio_venta,vendedor,fecha_venta,canal_venta
0,180325,Consola,2580484,Luis,2023-03-01,online
1,796559,Libro,48995,Pedro,2025-03-03,online
2,689113,Tablet,557920,Miguel,2025-05-11,online


In [9]:
print("\n" + "="*60)
print("ANÁLISIS COMPARATIVO")
print("="*60)

# Tabla de comparación
comparacion = []

if lectura_mono_exitosa and lectura_inc_exitosa:
    comparacion.append({
        "Métrica": "Filas cargadas",
        "Monolítica": f"{len(dataset_mono):,}",
        "Incremental": f"{filas_totales_inc:,}",
        "Coinciden": "✓" if len(dataset_mono) == filas_totales_inc else "✗"
    })
    
    comparacion.append({
        "Métrica": "Tiempo (seg)",
        "Monolítica": f"{tiempo_lectura_mono:.2f}",
        "Incremental": f"{tiempo_lectura_inc:.2f}",
        "Coinciden": f"{(tiempo_lectura_mono - tiempo_lectura_inc):.2f}s diferencia"
    })
    
    comparacion.append({
        "Métrica": "Memoria pico (MB)",
        "Monolítica": f"{peak / (1024 * 1024):.2f}",
        "Incremental": f"{peak_inc / (1024 * 1024):.2f}",
        "Coinciden": f"{((peak - peak_inc) / (1024 * 1024)):.2f}MB diferencia"
    })
    
    comparacion.append({
        "Métrica": "Velocidad (filas/seg)",
        "Monolítica": f"{len(dataset_mono) / tiempo_lectura_mono:,.0f}",
        "Incremental": f"{filas_totales_inc / tiempo_lectura_inc:,.0f}",
        "Coinciden": ""
    })
    
    df_comparacion = pd.DataFrame(comparacion)
    display(df_comparacion)
    
    print("\n📊 CONCLUSIONES:")
    print("-" * 60)
    print("✓ Ambos métodos completados exitosamente")
    print(f"✓ Ambas lecturas obtuvieron {len(dataset_mono):,} registros")
    
    ahorro_memoria = (peak - peak_inc) / (1024 * 1024)
    print(f"✓ Ahorro de memoria (incremental): {ahorro_memoria:.2f} MB ({100 * (1 - peak_inc/peak):.1f}%)")
    
    if tiempo_lectura_mono < tiempo_lectura_inc:
        print(f"✓ Lectura monolítica fue {tiempo_lectura_inc / tiempo_lectura_mono:.2f}x más rápida")
    else:
        print(f"✓ Lectura incremental fue {tiempo_lectura_mono / tiempo_lectura_inc:.2f}x más rápida")
    
    print("\n🎯 RECOMENDACIONES:")
    print("- Lectura monolítica: Ideal para archivos pequeños-medianos (<500 MB)")
    print("- Lectura incremental: Mejor para procesamiento en lotes o recursos limitados")
    
elif lectura_mono_exitosa and not lectura_inc_exitosa:
    print("⚠️  Lectura incremental falló pero monolítica fue exitosa")
    print(f"   Monolítica: {len(dataset_mono):,} filas en {tiempo_lectura_mono:.2f}s")
    
elif not lectura_mono_exitosa and lectura_inc_exitosa:
    print("⚠️  LÍMITE DETECTADO: Lectura monolítica falló")
    print(f"   Motivo: Memoria insuficiente o timeout")
    print(f"\n✓ Lectura incremental logró procesar {filas_totales_inc:,} filas con éxito")
    print(f"   Método incremental es OBLIGATORIO para este archivo")
    
else:
    print("✗ Ambas lecturas fallaron. Revisar archivo o permisos.")



ANÁLISIS COMPARATIVO


,Métrica,Monolítica,Incremental,Coinciden
0,Filas cargadas,"10,000,000","10,000,000",✓
1,Tiempo (seg),5.44,6.98,-1.54s diferencia
2,Memoria pico (MB),863.94,475.65,388.29MB diferencia
3,Velocidad (filas/seg),"1,839,851","1,433,045",



📊 CONCLUSIONES:
------------------------------------------------------------
✓ Ambos métodos completados exitosamente
✓ Ambas lecturas obtuvieron 10,000,000 registros
✓ Ahorro de memoria (incremental): 388.29 MB (44.9%)
✓ Lectura monolítica fue 1.28x más rápida

🎯 RECOMENDACIONES:
- Lectura monolítica: Ideal para archivos pequeños-medianos (<500 MB)
- Lectura incremental: Mejor para procesamiento en lotes o recursos limitados
